# 02 — Memakai LLM lewat API: Claude API dengan Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/08-llm-basics/02_claude_api_dasar.ipynb)

Di notebook 01 kita menjalankan LLM kecil secara lokal. Untuk aplikasi nyata, umumnya kita memakai LLM frontier lewat **API**: kirim permintaan HTTP berisi percakapan, terima jawaban. Tidak perlu GPU sama sekali; komputasi berjalan di server penyedia.

Notebook ini memakai **Claude API** dengan SDK resmi Python (`anthropic`). Isi:
1. Permintaan pertama dan **anatomi respons** (content blocks, `stop_reason`, `usage`).
2. **System prompt**.
3. **Prompt engineering** dasar: instruksi jelas, tag XML, few-shot, diukur pada tugas klasifikasi sentimen.
4. **Percakapan multi-giliran** (API bersifat *stateless*).
5. **Streaming**.
6. **Effort & thinking**: mengatur seberapa dalam model bernalar.
7. **Token & biaya**.
8. **Penanganan error** dan **refusal fallback**.

> Notebook ini tidak butuh GPU. Biaya API untuk menjalankan semuanya diperkirakan kurang dari US$1 (bagian 7 menghitungnya).

## 0. Instalasi dan API Key

Butuh **API key Claude** dari [Claude Console](https://platform.claude.com/) (menu *API keys*). Pemakaian API berbayar per token; biaya menjalankan notebook ini diperkirakan kurang dari US$1.

**Jangan pernah menulis API key langsung di kode atau notebook.** Siapa pun yang melihat notebook (atau riwayat git) bisa memakainya. Simpan di salah satu tempat berikut:
- **Colab:** ikon 🔑 (**Secrets**) di panel kiri → *Add new secret* bernama `ANTHROPIC_API_KEY` → aktifkan *Notebook access*.
- **Lokal:** salin `.env.example` di root repo menjadi `.env` lalu isi `ANTHROPIC_API_KEY=...`. File `.env` sudah diabaikan oleh `.gitignore`.

In [ ]:
%pip install -q "anthropic>=1.9" python-dotenv pandas

Sel berikut mencari API key (environment variable / `.env`, lalu Colab Secrets) dan membuat `client`. Konfigurasi lain:
- `MODEL`: `claude-opus-5-5`, model Claude yang paling mampu saat notebook ini ditulis.
- `FALLBACK`: mengaktifkan **refusal fallback**. Model Claude punya pengaman (safety classifier) yang bisa menolak permintaan tertentu. Dengan `fallbacks="default"`, jika permintaan ditolak, server otomatis mencoba ulang di model cadangan yang direkomendasikan Anthropic dalam panggilan yang sama, sehingga kode kita tidak perlu menangani pengulangan sendiri. Fitur ini masih **beta**, jadi dipanggil lewat `client.beta.messages` dengan header beta yang sesuai. Detailnya dibahas di bagian akhir notebook 02.

In [ ]:
import os

import anthropic
from dotenv import find_dotenv, load_dotenv

MODEL = "claude-opus-5-5"
FALLBACK = {"betas": ["server-side-fallback-2026-07-01"], "fallbacks": "default"}


def ambil_api_key():
    """Cari API key: environment variable / file .env (lokal), lalu Colab Secrets."""
    load_dotenv(find_dotenv(usecwd=True))          # baca .env terdekat (folder kerja atau induknya)
    if os.getenv("ANTHROPIC_API_KEY"):
        return os.getenv("ANTHROPIC_API_KEY")
    try:
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:  # bukan di Colab, atau Secret belum dibuat / belum diberi akses
        return None


api_key = ambil_api_key()
if not api_key:
    raise RuntimeError(
        "ANTHROPIC_API_KEY belum diatur.\n"
        "- Colab: ikon 🔑 Secrets → tambah ANTHROPIC_API_KEY → aktifkan Notebook access, lalu jalankan ulang sel ini.\n"
        "- Lokal: salin .env.example menjadi .env di root repo, lalu isi ANTHROPIC_API_KEY.")

client = anthropic.Anthropic(api_key=api_key)
print(f"✅ API key ditemukan | SDK anthropic {anthropic.__version__} | model: {MODEL}")

## 1. Permintaan Pertama

Endpoint utamanya adalah **Messages API**. Parameter wajib:
- `model`: nama model.
- `max_tokens`: batas **atas** panjang jawaban (termasuk token *thinking*). Ini batas, bukan target; kamu hanya membayar token yang benar-benar dihasilkan.
- `messages`: daftar pesan `{"role": "user" | "assistant", "content": ...}`, dimulai dari `user`.

In [ ]:
respon = client.beta.messages.create(
    model=MODEL,
    max_tokens=16000,
    messages=[{"role": "user", "content": "Jelaskan apa itu large language model dalam 2 kalimat."}],
    **FALLBACK,
)

for blok in respon.content:
    if blok.type == "text":
        print(blok.text)

**Anatomi respons.** `content` adalah **daftar blok**, bukan satu string. Selain `text`, bisa ada blok `thinking` (penalaran model), `tool_use` (notebook 03), dan lainnya. Karena itu, selalu periksa `blok.type` sebelum membaca `.text`.

- `stop_reason`: alasan model berhenti. `end_turn` = selesai normal, `max_tokens` = terpotong, `tool_use` = ingin memanggil tool, `refusal` = menolak.
- `usage`: jumlah token input & output. Inilah dasar tagihan.
- `_request_id`: ID permintaan, berguna saat melapor masalah ke Anthropic.

In [ ]:
print("id          :", respon.id)
print("model       :", respon.model)
print("stop_reason :", respon.stop_reason)
print("jenis blok  :", [b.type for b in respon.content])
print("usage       : input", respon.usage.input_tokens, "token | output", respon.usage.output_tokens, "token")
print("request id  :", respon._request_id)

Agar sel-sel berikutnya ringkas, kita buat dua helper:
- `teks(respon)`: mengambil gabungan blok teks.
- `tanya(...)`: memanggil API dengan setelan standar, memberi peringatan jika jawaban terpotong atau ditolak, dan mencatat biaya (bagian 7).

In [ ]:
def teks(respon):
    """Gabungkan semua blok teks dari respons (respons juga bisa berisi blok thinking, tool_use, dll.)."""
    return "".join(b.text for b in respon.content if b.type == "text")


# Harga claude-opus-5-5 (USD per 1 juta token). Harga bisa berubah: cek halaman harga resmi Anthropic.
HARGA_PER_JUTA = {"input": 4.00, "output": 20.00}
catatan_biaya = []


def biaya(usage):
    """Perkiraan biaya (USD) dari objek usage. Token thinking dihitung sebagai token output."""
    return (usage.input_tokens * HARGA_PER_JUTA["input"] + usage.output_tokens * HARGA_PER_JUTA["output"]) / 1e6


def tanya(pesan, system=None, max_tokens=16000, **kwargs):
    """Panggil Messages API. `pesan` boleh string (satu pesan user) atau daftar messages."""
    if isinstance(pesan, str):
        pesan = [{"role": "user", "content": pesan}]
    if system:
        kwargs["system"] = system
    respon = client.beta.messages.create(model=MODEL, max_tokens=max_tokens, messages=pesan, **FALLBACK, **kwargs)
    if respon.stop_reason == "refusal":
        print("⚠️ Permintaan ditolak model:", respon.stop_details)
    elif respon.stop_reason == "max_tokens":
        print("⚠️ Jawaban terpotong karena mencapai max_tokens")
    catatan_biaya.append(biaya(respon.usage))
    return respon


print(teks(tanya("Sebutkan 3 contoh penggunaan LLM di e-commerce, masing-masing satu kalimat.")))

## 2. System Prompt

**System prompt** memberi konteks dan aturan yang berlaku sepanjang percakapan: peran, gaya bahasa, format, batasan. Ia dikirim lewat parameter `system`, terpisah dari `messages`.

Bandingkan jawaban untuk pertanyaan yang sama tanpa dan dengan system prompt:

In [ ]:
pertanyaan = "Apa itu backpropagation?"
SYSTEM_TUTOR = (
    "Kamu tutor deep learning untuk pemula yang sudah paham Python dan ML dasar. "
    "Jawab dalam bahasa Indonesia santai, maksimal 4 kalimat, dan selalu sertakan satu analogi sehari-hari."
)

print("=== Tanpa system prompt ===")
print(teks(tanya(pertanyaan)))
print("\n=== Dengan system prompt ===")
print(teks(tanya(pertanyaan, system=SYSTEM_TUTOR)))

## 3. Prompt Engineering: Diukur, Bukan Ditebak

Prompt engineering = merancang input agar output sesuai kebutuhan. Prinsip dasarnya:
1. **Jelas dan spesifik**: sebutkan tugas, definisi label, dan format output yang diharapkan.
2. **Pisahkan instruksi dan data** dengan tag XML (misalnya `<ulasan>...</ulasan>`) agar model tidak bingung mana perintah dan mana data.
3. **Few-shot**: beri beberapa contoh input → output yang benar.

Seperti di ML, perubahan prompt sebaiknya **diukur** pada data berlabel, bukan dinilai dari satu-dua contoh. Kita pakai 8 ulasan produk berbahasa Indonesia (ditulis untuk notebook ini) dengan label sentimen.

In [ ]:
import pandas as pd

ULASAN = [
    ("Pengiriman cepat, barang sesuai deskripsi. Mantap!", "positif"),
    ("Baterai cepat habis, baru dua hari sudah drop.", "negatif"),
    ("Barang sudah sampai, belum dicoba.", "netral"),
    ("Gila sih, kamera malamnya keren banget.", "positif"),
    ("Katanya garansi resmi, ternyata garansi distributor. Kecewa berat.", "negatif"),
    ("Admin lambat balas chat, tapi barangnya bagus.", "netral"),
    ("Casing retak di bagian sudut, untung layarnya aman.", "negatif"),
    ("Sesuai foto. Ukurannya pas.", "positif"),
]
LABEL = {"positif", "negatif", "netral"}
print(f"{len(ULASAN)} ulasan")

Tiga versi prompt:
- **A. Seadanya**: hanya pertanyaan.
- **B. Jelas + XML**: definisi label, data dalam tag, format output tegas.
- **C. B + few-shot**: ditambah contoh percakapan (pasangan `user`/`assistant`) sebelum pertanyaan sebenarnya.

Klasifikasi sentimen adalah tugas sederhana, jadi kita memakai `effort="low"` agar model tidak bernalar panjang (lebih cepat dan murah; dibahas di bagian 6).

In [ ]:
SYSTEM_SENTIMEN = """Kamu mengklasifikasikan sentimen ulasan produk berbahasa Indonesia.
Definisi label:
- positif: pembeli puas secara keseluruhan.
- negatif: ada keluhan utama tentang produk atau layanan.
- netral: tidak ada opini, atau pujian dan keluhan sama kuat.
Jawab HANYA dengan satu kata: positif, negatif, atau netral."""

CONTOH_FEW_SHOT = [
    {"role": "user", "content": "<ulasan>Produk original, packing rapi, seller ramah.</ulasan>"},
    {"role": "assistant", "content": "positif"},
    {"role": "user", "content": "<ulasan>Pesanan datang, nanti saya kabari lagi.</ulasan>"},
    {"role": "assistant", "content": "netral"},
    {"role": "user", "content": "<ulasan>Warnanya cantik, tapi jahitannya lepas setelah sekali cuci.</ulasan>"},
    {"role": "assistant", "content": "negatif"},
]


def prompt_a(u):
    return tanya(f"Apa sentimen ulasan ini? {u}", output_config={"effort": "low"})


def prompt_b(u):
    return tanya(f"<ulasan>{u}</ulasan>", system=SYSTEM_SENTIMEN, output_config={"effort": "low"})


def prompt_c(u):
    pesan = CONTOH_FEW_SHOT + [{"role": "user", "content": f"<ulasan>{u}</ulasan>"}]
    return tanya(pesan, system=SYSTEM_SENTIMEN, output_config={"effort": "low"})


hasil = []
for nama, fungsi in [("A. seadanya", prompt_a), ("B. jelas + XML", prompt_b), ("C. few-shot", prompt_c)]:
    for u, label in ULASAN:
        jawaban = teks(fungsi(u)).strip()
        normal = jawaban.lower().strip(" .!\n")
        hasil.append({"prompt": nama, "ulasan": u, "label": label, "jawaban": jawaban[:60],
                      "format_valid": normal in LABEL, "benar": normal == label})

df_prompt = pd.DataFrame(hasil)
print(df_prompt.groupby("prompt")[["format_valid", "benar"]].mean().round(2))

Lihat jawaban mentah prompt A: biasanya berupa kalimat penjelasan, sehingga tidak bisa langsung dipakai program (format tidak valid), walaupun isinya mungkin benar. Prompt B dan C memberi jawaban satu kata yang konsisten.

Ulasan campuran (misalnya "Admin lambat balas chat, tapi barangnya bagus") adalah kasus sulit: labelnya bergantung pada **definisi** yang kita tetapkan. Inilah fungsi definisi label dan contoh few-shot: menyamakan "selera" model dengan kebutuhan kita.

In [ ]:
df_prompt[df_prompt["prompt"] == "A. seadanya"][["ulasan", "label", "jawaban"]].head(4)

In [ ]:
df_prompt[~df_prompt["benar"] & (df_prompt["prompt"] != "A. seadanya")][["prompt", "ulasan", "label", "jawaban"]]

## 4. Percakapan Multi-Giliran

API bersifat **stateless**: server tidak menyimpan percakapan. Agar model "ingat", setiap permintaan harus membawa **seluruh riwayat**: pesan user, jawaban assistant sebelumnya, lalu pesan baru.

Jawaban assistant ditambahkan ke riwayat sebagai `respon.content` (daftar blok lengkap, termasuk blok thinking), bukan hanya teksnya. Dengan begitu penalaran model di giliran sebelumnya ikut terbawa.

In [ ]:
riwayat = []


def ngobrol(pesan_user, system=SYSTEM_TUTOR):
    riwayat.append({"role": "user", "content": pesan_user})
    respon = tanya(riwayat, system=system)
    riwayat.append({"role": "assistant", "content": respon.content})
    print(f"🧑 {pesan_user}\n🤖 {teks(respon)}")
    print(f"   (token input giliran ini: {respon.usage.input_tokens})\n")


ngobrol("Saya ingin latihan CNN. Rekomendasikan satu dataset gambar kecil.")
ngobrol("Kenapa dataset itu cocok untuk pemula?")
ngobrol("Oke, kalau sudah bisa, dataset apa yang sebaiknya saya coba setelahnya?")

Perhatikan **token input naik setiap giliran** karena riwayat makin panjang. Percakapan panjang = biaya per giliran makin mahal, dan suatu saat bisa melebihi context window. Solusi umum: meringkas riwayat lama, membuang giliran yang tidak relevan, atau memakai fitur *prompt caching* dan *compaction* dari API.

Buktinya, pertanyaan kedua tanpa riwayat tidak bisa dijawab dengan tepat karena model tidak tahu "dataset itu" merujuk ke mana:

In [ ]:
print(teks(tanya("Kenapa dataset itu cocok untuk pemula?", system=SYSTEM_TUTOR)))

## 5. Streaming

Tanpa streaming, kita menunggu sampai seluruh jawaban selesai. Dengan **streaming**, token dikirim segera setelah dibuat, sehingga pengguna langsung melihat teks muncul (seperti di aplikasi chat). Streaming juga dianjurkan untuk jawaban panjang agar koneksi tidak kena *timeout*.

`stream.text_stream` menghasilkan potongan teks; `get_final_message()` mengembalikan respons lengkap (sama bentuknya dengan hasil `create`).

In [ ]:
with client.beta.messages.stream(
    model=MODEL,
    max_tokens=16000,
    system=SYSTEM_TUTOR,
    messages=[{"role": "user", "content": "Ceritakan secara singkat perkembangan AI dari perceptron sampai LLM."}],
    **FALLBACK,
) as stream:
    for potongan in stream.text_stream:
        print(potongan, end="", flush=True)
    final = stream.get_final_message()

catatan_biaya.append(biaya(final.usage))
print(f"\n\nstop_reason: {final.stop_reason} | token output: {final.usage.output_tokens}")

## 6. Effort dan Thinking

Model Claude terbaru **berpikir dulu** sebelum menjawab (*adaptive thinking*): model menentukan sendiri seberapa panjang penalarannya sesuai tingkat kesulitan soal. Di `claude-opus-5-5` thinking selalu aktif; yang kita atur adalah **effort** lewat `output_config={"effort": ...}`:

| effort | Kapan dipakai |
|---|---|
| `low` | Tugas sederhana & volume tinggi (klasifikasi, ekstraksi). Cepat dan murah. |
| `medium` | Default model ini. Seimbang. |
| `high`, `xhigh`, `max` | Soal sulit: penalaran bertingkat, matematika, kode kompleks. Lebih lambat dan lebih mahal. |

Token thinking dihitung sebagai **token output** (ditagih). Secara default isi thinking tidak ditampilkan; `thinking={"type": "adaptive", "display": "summarized"}` meminta ringkasannya.

In [ ]:
import time

TEKA_TEKI = (
    "Ada tiga kotak berlabel 'apel', 'jeruk', dan 'campur'. Semua label PASTI salah. "
    "Kamu boleh mengambil satu buah dari satu kotak saja tanpa melihat isinya. "
    "Bagaimana cara menentukan isi ketiga kotak dengan benar? Jawab ringkas."
)

baris = []
for effort in ["low", "high"]:
    mulai = time.perf_counter()
    r = tanya(TEKA_TEKI, thinking={"type": "adaptive", "display": "summarized"}, output_config={"effort": effort})
    ringkasan_thinking = " ".join(b.thinking for b in r.content if b.type == "thinking")
    baris.append({"effort": effort, "detik": round(time.perf_counter() - mulai, 1),
                  "token output": r.usage.output_tokens, "biaya (USD)": round(biaya(r.usage), 4)})
    print(f"=== effort={effort} ===")
    print("💭 thinking (ringkasan):", (ringkasan_thinking[:300] + "…") if ringkasan_thinking else "(kosong)")
    print("💬 jawaban:", teks(r), "\n")

pd.DataFrame(baris)

Untuk teka-teki ini keduanya kemungkinan besar benar, tetapi effort tinggi memakai lebih banyak token (dan waktu). Pilih effort serendah mungkin yang masih memberi kualitas cukup, dan ukur pada data (seperti bagian 3).

**Bagaimana dengan temperature/top-p?** Di notebook 01 kita mempelajari sampling pada model lokal. Model Claude terbaru mengatur sampling sendiri; SDK `anthropic` 1.x bahkan tidak lagi menyediakan parameternya. Perilaku model diarahkan lewat prompt dan effort:

In [ ]:
import inspect

parameter = inspect.signature(client.beta.messages.create).parameters
for p in ["temperature", "top_p", "top_k", "system", "output_config", "thinking"]:
    print(f"{p:14}: {'tersedia' if p in parameter else 'tidak tersedia'}")

## 7. Token dan Biaya

Biaya = token input × harga input + token output × harga output. Untuk `claude-opus-5-5`: US$4 per 1 juta token input dan US$20 per 1 juta token output (cek halaman harga resmi; harga bisa berubah). Token output 5× lebih mahal, dan token thinking termasuk output.

`count_tokens` menghitung token input **sebelum** mengirim permintaan (gratis), berguna untuk memperkirakan biaya atau mengecek batas context window. Kita bandingkan lagi teks Indonesia vs Inggris, kali ini dengan tokenizer Claude:

In [ ]:
def hitung_token(isi):
    return client.beta.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": isi}]).input_tokens


indo = "Pengiriman barangnya cepat sekali dan kualitasnya memuaskan, pasti beli lagi di toko ini."
inggris = "The delivery was very fast and the quality is satisfying, I will definitely buy again from this store."
for nama, isi in [("Indonesia", indo), ("Inggris", inggris)]:
    print(f"{nama:9}: {hitung_token(isi)} token input ({len(isi.split())} kata)")

Angka di atas sudah termasuk beberapa token pembungkus format pesan, jadi fokus pada selisihnya. Sekarang total biaya semua panggilan lewat `tanya()` dan streaming di notebook ini, serta proyeksi untuk skala produksi:

In [ ]:
total = sum(catatan_biaya)
print(f"{len(catatan_biaya)} panggilan API | total perkiraan biaya: US${total:.4f}")

# Proyeksi: klasifikasi sentimen (prompt C) untuk 100.000 ulasan
r = prompt_c("Barangnya bagus, pengiriman agak lama.")
print(f"Satu klasifikasi: {r.usage.input_tokens} token input + {r.usage.output_tokens} token output "
      f"= US${biaya(r.usage):.5f} → 100.000 ulasan ≈ US${biaya(r.usage) * 100_000:,.0f}")

Proyeksi seperti ini penting sebelum membawa LLM ke produksi. Cara menekan biaya: memakai model yang lebih kecil/murah untuk tugas sederhana, effort rendah, prompt caching untuk system prompt dan contoh few-shot yang selalu sama, dan **Batch API** (diskon untuk pekerjaan yang tidak harus real-time).

## 8. Penanganan Error

Panggilan API bisa gagal karena berbagai alasan. SDK mengubah setiap status HTTP menjadi exception yang spesifik, sehingga kita bisa menanganinya dengan tepat:

| Exception | Status | Penyebab umum | Tindakan |
|---|---|---|---|
| `BadRequestError` | 400 | Parameter/format permintaan salah | Perbaiki kode, jangan diulang |
| `AuthenticationError` | 401 | API key salah atau dicabut | Periksa key |
| `NotFoundError` | 404 | Nama model salah | Periksa nama model |
| `RateLimitError` | 429 | Terlalu banyak permintaan | Tunggu lalu ulangi |
| `InternalServerError` / 5xx | 500+ | Gangguan server | Ulangi nanti |
| `APIConnectionError` | – | Jaringan bermasalah | Periksa koneksi, ulangi |

Kabar baiknya: SDK **otomatis mengulang** error 429, 5xx, dan error koneksi (default 2 kali, dengan *exponential backoff*).

In [ ]:
def coba(nama, fungsi):
    try:
        fungsi()
        print(f"✅ {nama}: berhasil")
    except anthropic.BadRequestError as e:
        print(f"❌ {nama}: BadRequestError ({e.status_code}) → {e.message}")
    except anthropic.AuthenticationError as e:
        print(f"❌ {nama}: AuthenticationError ({e.status_code}) → API key salah")
    except anthropic.NotFoundError as e:
        print(f"❌ {nama}: NotFoundError ({e.status_code}) → {e.message}")
    except anthropic.RateLimitError:
        print(f"❌ {nama}: RateLimitError → tunggu lalu ulangi")
    except anthropic.APIStatusError as e:
        print(f"❌ {nama}: APIStatusError ({e.status_code}) → {e.message}")
    except anthropic.APIConnectionError:
        print(f"❌ {nama}: APIConnectionError → periksa koneksi internet")


halo = [{"role": "user", "content": "Halo"}]
coba("Permintaan normal", lambda: client.beta.messages.create(model=MODEL, max_tokens=100, messages=halo))
coba("Nama model salah", lambda: client.beta.messages.create(model="claude-model-tidak-ada", max_tokens=100, messages=halo))
coba("messages kosong", lambda: client.beta.messages.create(model=MODEL, max_tokens=100, messages=[]))
coba("API key salah", lambda: anthropic.Anthropic(api_key="kunci-salah", max_retries=0).beta.messages.create(
    model=MODEL, max_tokens=100, messages=halo))

Jumlah pengulangan dan batas waktu bisa diatur per client atau per permintaan dengan `with_options`:

In [ ]:
r = client.with_options(max_retries=5, timeout=60).beta.messages.create(
    model=MODEL, max_tokens=1000, messages=[{"role": "user", "content": "Balas dengan satu kata: siap?"}], **FALLBACK)
print(teks(r))

### Refusal dan fallback

Kadang model **menolak** permintaan, misalnya karena safety classifier menilai permintaan berisiko. Penolakan **bukan** exception: responsnya tetap HTTP 200, tetapi `stop_reason == "refusal"` dan `stop_details` berisi kategori alasannya. Jadi periksa `stop_reason` sebelum memakai isi respons (helper `tanya()` sudah melakukannya).

Karena kita mengirim `FALLBACK` (`fallbacks="default"`), permintaan yang ditolak otomatis dijalankan ulang oleh server di model cadangan. Jika itu terjadi, respons memuat blok `fallback` yang mencatat perpindahannya, dan `respon.model` berisi model yang benar-benar menjawab. Jika model cadangan pun menolak, `stop_reason` tetap `refusal`.

Pertanyaan belajar biasa hampir tidak pernah ditolak, jadi di bawah ini kemungkinan besar tidak ada fallback. Fungsinya tetap berguna untuk dipasang di aplikasi.

In [ ]:
def info_fallback(respon):
    for b in respon.content:
        if b.type == "fallback":
            print(f"↪️ {b.from_.model} menolak; dilanjutkan oleh {b.to.model}")
    print(f"stop_reason: {respon.stop_reason} | dijawab oleh: {respon.model}")


info_fallback(tanya("Apa perbedaan supervised dan unsupervised learning? Satu kalimat saja."))

## Ringkasan

- API key disimpan di **Colab Secrets** atau **`.env`**, tidak pernah di kode.
- Respons berisi **daftar blok** (`text`, `thinking`, `tool_use`, ...). Selalu cek `stop_reason` (`end_turn`, `max_tokens`, `refusal`, ...) dan `usage`.
- **System prompt** mengatur peran & aturan; **prompt engineering** (instruksi jelas, tag XML, few-shot) sebaiknya **diukur** pada data berlabel.
- API **stateless**: kirim ulang seluruh riwayat setiap giliran, sehingga token input terus bertambah.
- **Streaming** untuk respons cepat terlihat dan jawaban panjang.
- **Effort** mengatur kedalaman penalaran (dan biaya); sampling diatur model sendiri.
- Biaya dihitung per token; gunakan `count_tokens` dan proyeksi sebelum produksi.
- Tangani exception SDK sesuai jenisnya; aktifkan **refusal fallback** dan periksa `stop_reason`.

## Latihan

1. **Chatbot tutor dengan batas riwayat.** Buat kelas `Tutor` yang menyimpan riwayat dan hanya mengirim `N` giliran terakhir (plus system prompt). Jalankan 8 giliran dan buat grafik token input per giliran untuk `N=None` (semua riwayat) vs `N=4`. Apa risikonya jika riwayat dipotong?
2. **Evaluasi prompt yang lebih serius.** Unduh 60 kalimat NusaX-Senti bahasa Indonesia (lihat modul 06) dan bandingkan prompt B vs C, masing-masing dengan `effort="low"` dan `"medium"`. Laporkan akurasi, macro-F1, dan biaya per 1.000 ulasan. Bandingkan dengan hasil fine-tune IndoBERT di modul 06: kapan lebih masuk akal memakai LLM lewat API, kapan model kecil hasil fine-tune?
3. **Chat di terminal dengan streaming.** Tulis skrip Python (bukan notebook) `chat.py` yang membaca API key dari `.env`, menerima input pengguna berulang kali dengan `input()`, menampilkan jawaban secara streaming, mendukung perintah `/reset` (hapus riwayat) dan `/biaya` (total biaya sesi), serta menyimpan percakapan ke file JSON saat keluar.